In [32]:

# 사전설치 : pip install pandas, numpy, matplotlib, seaborn, sklearn, xgboost
import math
import pickle
from typing import Dict, List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')


In [40]:

SCRIPT_DIR = Path('./신재용_머신러닝프로젝트.ipynb').resolve().parent

def find_base_dir() -> Path:
  candidates = [SCRIPT_DIR, SCRIPT_DIR.parent, SCRIPT_DIR.parent.parent]
  for candidate in candidates:
    if (candidate / "dataset").exists():
      return candidate
  return SCRIPT_DIR

BASE_DIR = find_base_dir()
REFORMED_DIR = BASE_DIR / "dataset" / "reformed_data"
MODEL_OUTPUT_DIR = BASE_DIR / "dataset/model_output"

In [ ]:
#데이터 로드
try:
 print(REFORMED_DIR)
 X_train = pd.read_csv(REFORMED_DIR / "X_train.csv")
 X_val = pd.read_csv(REFORMED_DIR / "X_val.csv")
 X_test = pd.read_csv(REFORMED_DIR / "X_test.csv")

 y_train = pd.read_csv(REFORMED_DIR / "y_train.csv").iloc[:, 0]
 y_val = pd.read_csv(REFORMED_DIR / "y_val.csv").iloc[:, 0]
 y_test = pd.read_csv(REFORMED_DIR / "y_test.csv").iloc[:, 0]

 train_df = pd.read_csv(REFORMED_DIR / "train_df.csv")
 val_df = pd.read_csv(REFORMED_DIR / "val_df.csv")
 test_df = pd.read_csv(REFORMED_DIR / "test_df.csv")
 train_df["date"] = pd.to_datetime(train_df["date"])
 val_df["date"] = pd.to_datetime(val_df["date"])
 test_df["date"] = pd.to_datetime(test_df["date"])

 date_origin = pd.to_datetime("2024-01-01")
 print('데이터 로드 완료')
 
except Exception as e:
 print(f"전처리된 csv파일이 없습니다.\n{e}")

D:\git\project_week2\dataset\reformed_data
데이터 로드 완료


In [46]:
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import r2_score, mean_absolute_error


#모델 비교 & 선택
specs = [
    {
        "name": "linear_regression_log",
        "factory": lambda: LinearRegression(),
        "use_log_target": True,
    },
    {
        "name": "ridge_alpha_1_log",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]),
        "use_log_target": True,
    },
    {
        "name": "ridge_alpha_10_log",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0)),
        ]),
        "use_log_target": True,
    },
    {
        "name": "random_forest_log",
        "factory": lambda: RandomForestRegressor(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        "use_log_target": True,
    },
    {
        "name": "gradient_boosting_log",
        "factory": lambda: GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            random_state=42,
        ),
        "use_log_target": True,
    },
]
print("각 모델 생성 완료")

rows = []
fitted_models = {}

#모델 예상치 추출 함수
def predict_candidate_model(spec, model, X: pd.DataFrame) -> np.ndarray:
    pred = model.predict(X)
    if spec["use_log_target"]:
        pred = np.expm1(pred)
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    return pred
#모델 성능 수치 반환 함수
def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    safe_true = np.where(y_true == 0, np.nan, y_true)
    mape = np.nanmean(np.abs((y_true - y_pred) / safe_true)) * 100
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(mape) if not math.isnan(mape) else np.nan,
        "bias": float(np.mean(y_pred - y_true)),
    }

for spec in specs : 
    model = spec['factory']()
    if spec['use_log_target']:
        model.fit(X_train, np.log1p(y_train))
    else:
        model.fit(X_train, y_train)
    fitted_models[spec['name']] = (spec, model)
    #모델 예상치 추출
    train_pred = predict_candidate_model(spec, model, X_train)
    val_pred = predict_candidate_model(spec, model, X_val)
    test_pred = predict_candidate_model(spec, model, X_test)
    #모델 평가
    train_metrics = calc_regression_metrics(y_train, train_pred)
    val_metrics = calc_regression_metrics(y_val, val_pred)
    test_metrics = calc_regression_metrics(y_test, test_pred)

    rows.append({
        "model_name": spec["name"],
        "train_r2": train_metrics["r2"],
        "val_r2": val_metrics["r2"],
        "test_r2": test_metrics["r2"],
        "train_mae": train_metrics["mae"],
        "val_mae": val_metrics["mae"],
        "test_mae": test_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "val_rmse": val_metrics["rmse"],
        "test_rmse": test_metrics["rmse"],
        "val_mape": val_metrics["mape"],
        "test_mape": test_metrics["mape"],
        "val_bias": val_metrics["bias"],
        "test_bias": test_metrics["bias"],
    })

print('모델 평가 및 결과 저장 완료')


comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df.sort_values(by=["val_r2", "val_mae"], ascending=[False, True]).reset_index(drop=True)
comparison_df.to_csv(MODEL_OUTPUT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

best_name = comparison_df.iloc[0]["model_name"]
best_spec, best_eval_model = fitted_models[best_name]

X_train_val = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# 최종 배포용 모델은 train + val 전체로 다시 학습
best_inference_model = best_spec["factory"]()

if best_spec["use_log_target"]:
    best_inference_model.fit(X_train_val, np.log1p(y_train_val))
else:
    best_inference_model.fit(X_train_val, y_train_val)
    
print("최종 모델 학습 완료")
# model = GradientBoostingRegressor(
#  n_estimators=300,
#  learning_rate=0.05,
#  max_depth=3,
#  subsample=0.9,
#  random_state=42,
#  )

# model.fit(X_train, y_train)

각 모델 생성 완료
모델 평가 및 결과 저장 완료
최종 모델 학습 완료


In [47]:
#모델 평가 & 결과 저장
def save_prediction_results(prefix_df: pd.DataFrame, y_true, y_pred, filename: str):
    out = pd.DataFrame({
        "date": prefix_df["date"],
        "actual": y_true,
        "predicted": y_pred,
    })
    out.to_csv(MODEL_OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")
    return out

def save_scatter_plot(y_true, y_pred, title: str, filename: str, show_plot: bool):
    plt.figure(figsize=(10, 6))
    plt.scatter(y_true, y_pred, alpha=0.55)
    min_val = min(np.min(y_true), np.min(y_pred))
    max_val = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2)
    plt.xlabel("실제 대여건수")
    plt.ylabel("예측 대여건수")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(MODEL_OUTPUT_DIR / filename, dpi=150)
    if show_plot:
        plt.show()
    else:
        plt.close()


def save_timeseries_plot(result_df: pd.DataFrame, title: str, filename: str, show_plot: bool):
    plt.figure(figsize=(12, 6))
    plt.plot(result_df["date"], result_df["actual"], label="실제 대여건수")
    plt.plot(result_df["date"], result_df["predicted"], label="예측 대여건수")
    plt.xlabel("날짜")
    plt.ylabel("대여건수")
    plt.title(title)
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(MODEL_OUTPUT_DIR / filename, dpi=150)
    if show_plot:
        plt.show()
    else:
        plt.close()
        

train_pred = predict_candidate_model(best_spec, best_eval_model, X_train)
val_pred = predict_candidate_model(best_spec, best_eval_model, X_val)
test_pred = predict_candidate_model(best_spec, best_eval_model, X_test)

train_metrics = calc_regression_metrics(y_train, train_pred)
val_metrics = calc_regression_metrics(y_val, val_pred)
test_metrics = calc_regression_metrics(y_test, test_pred)

train_result = save_prediction_results(train_df, y_train, train_pred, "train_prediction_result.csv")
val_result = save_prediction_results(val_df, y_val, val_pred, "val_prediction_result.csv")
test_result = save_prediction_results(test_df, y_test, test_pred, "test_prediction_result.csv")


save_scatter_plot(y_val, val_pred, f"Validation 실제값 vs 예측값 산점도 ({best_name})", "val_scatter.png", False)
save_scatter_plot(y_test, test_pred, f"Test 실제값 vs 예측값 산점도 ({best_name})", "test_scatter.png", False)
save_timeseries_plot(val_result, f"Validation 날짜별 실제값 vs 예측값 ({best_name})", "val_timeseries.png", False)
save_timeseries_plot(test_result, f"Test 날짜별 실제값 vs 예측값 ({best_name})", "test_timeseries.png", False)

In [48]:
#모델 저장
metadata = {
    "model_name": best_name,
    "feature_columns": X_train.columns.tolist(),
    "date_origin": date_origin,
    "use_log_target": best_spec["use_log_target"],
    "metrics": {
        "train": train_metrics,
        "validation": val_metrics,
        "test": test_metrics,
    },
    "comparison_df_path": str(MODEL_OUTPUT_DIR / "model_comparison.csv"),
    "feature_importance_path": str(MODEL_OUTPUT_DIR / "best_model_feature_importance.csv"),
    "metrics_csv_path": str(MODEL_OUTPUT_DIR / "best_model_metrics.csv"),
}

model_path = MODEL_OUTPUT_DIR / "best_model.joblib"   # 파일
metadata_path = MODEL_OUTPUT_DIR / "metadata.pkl"     # 파일

joblib.dump(best_inference_model, model_path)
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print("\n[학습 완료]")
print(f"- 선택 모델: {best_name}")
print(f"- Validation R²: {val_metrics['r2']:.4f}")
print(f"- Test R²: {test_metrics['r2']:.4f}")
print(f"- 모델 저장(joblib): {model_path}")
print(f"- 메타데이터 저장(pkl): {metadata_path}")


[학습 완료]
- 선택 모델: gradient_boosting_log
- Validation R²: 0.7123
- Test R²: 0.9194
- 모델 저장(joblib): D:\git\project_week2\dataset\model_output\best_model.joblib
- 메타데이터 저장(pkl): D:\git\project_week2\dataset\model_output\metadata.pkl
